In [2]:
import numpy as np

encoder_input = np.load('encoder_input.npy')
decoder_input = np.load('decoder_input.npy')
decoder_output = np.load('decoder_output.npy')

In [3]:
import joblib

idx2word = joblib.load('idx2word.pkl')
word2idx = joblib.load('word2idx.pkl')

In [4]:
from sklearn.model_selection import train_test_split

encoder_input_train, encoder_input_test, decoder_input_train, decoder_input_test, decoder_output_train, decoder_output_test = train_test_split(encoder_input, decoder_input, decoder_output, test_size=0.2, random_state=42)

In [5]:
vocab_size = len(word2idx)
vocab_size

26629

In [6]:
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, GRU, Dense, Embedding

encoder_inputs = Input(shape=(None,))
encoder_embedding = Embedding(vocab_size, 256)(encoder_inputs)
encoder_gru = GRU(256, return_state=True, return_sequences=True)
encoder_outputs, state_h = encoder_gru(encoder_embedding)
encoder_states = state_h

In [7]:
decoder_inputs = Input(shape=(None,))
decoder_embedding = Embedding(vocab_size, 256)(decoder_inputs)
decoder_gru = GRU(256, return_sequences=True, return_state=True)
decoder_outputs, _ = decoder_gru(decoder_embedding, initial_state=encoder_states)

In [15]:
from tensorflow.keras.layers import Attention

attention = Attention()
context_vector = attention([decoder_outputs, encoder_outputs])

In [16]:
from tensorflow.keras.layers import Concatenate, Dense

decoder_combined_context = Concatenate(axis=-1)([decoder_outputs, context_vector])
decoder_dense = Dense(vocab_size, activation='softmax')
decoder_prediction = decoder_dense(decoder_combined_context)

In [22]:
model = Model([encoder_inputs, decoder_inputs], decoder_prediction)
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=["accuracy"])
model.summary()

Model: "functional_2"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, None)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer_1       │ (None, None)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding           │ (None, None, 256) │  6,817,024 │ input_layer[0][0] │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_1         │ (None, None, 256) │  6,817,024 │ input_layer_1[0]… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ gru (GRU)           │ [(None, None,     │    394,752 │ embedding[0][0]   │
│                     │ 256), (None,      │            │                   │
│                     │ 256)]             │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ gru_1 (GRU)         │ [(None, None,     │    394,752 │ embedding_1[0][0… │
│                     │ 256), (None,      │            │ gru[0][1]         │
│                     │ 256)]             │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ attention_1         │ (None, None, 256) │          0 │ gru_1[0][0],      │
│ (Attention)         │                   │            │ gru[0][0]         │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate_1       │ (None, None, 512) │          0 │ gru_1[0][0],      │
│ (Concatenate)       │                   │            │ attention_1[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, None,      │ 13,660,677 │ concatenate_1[0]… │
│                     │ 26629)            │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 28,084,229 (107.13 MB)

 Trainable params: 28,084,229 (107.13 MB)

 Non-trainable params: 0 (0.00 B)

In [18]:
import numpy as np

decoder_output_train_expanded = np.expand_dims(decoder_output_train, -1)
decoder_output_test_expanded = np.expand_dims(decoder_output_test, -1)

In [19]:
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

early_stop = EarlyStopping(
    monitor="val_loss",
    patience=3,
    restore_best_weights=True,
)

reduce_lr = ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.5,
    patience=2,
    min_lr=1e-6,
)

In [23]:
history = model.fit(
    [encoder_input_train, decoder_input_train],
    decoder_output_train_expanded,
    validation_data=([encoder_input_test, decoder_input_test], decoder_output_test_expanded),
    batch_size=128,
    epochs=15,
    callbacks=[early_stop, reduce_lr],
)

Epoch 1/15
1251/1251 ━━━━━━━━━━━━━━━━━━━━ 558s 443ms/step - accuracy: 0.7828 - loss: 1.3248 - val_accuracy: 0.7807 - val_loss: 1.3810 - learning_rate: 0.0010
Epoch 2/15
1251/1251 ━━━━━━━━━━━━━━━━━━━━ 537s 425ms/step - accuracy: 0.7858 - loss: 1.2442 - val_accuracy: 0.7805 - val_loss: 1.3839 - learning_rate: 0.0010
Epoch 3/15
1251/1251 ━━━━━━━━━━━━━━━━━━━━ 561s 449ms/step - accuracy: 0.7898 - loss: 1.1635 - val_accuracy: 0.7793 - val_loss: 1.4070 - learning_rate: 0.0010
Epoch 4/15
1251/1251 ━━━━━━━━━━━━━━━━━━━━ 531s 425ms/step - accuracy: 0.7992 - loss: 1.0467 - val_accuracy: 0.7782 - val_loss: 1.4358 - learning_rate: 5.0000e-04
